## Libraries Installation

In [76]:
!pip install azure-storage-blob
!pip install snowflake-connector-python
!pip install snowflake-sqlalchemy
!pip install pandas
!pip install requests

## Libraries Import

In [77]:
import pandas as pd
import numpy as np
import os
import re
import json
import io
import warnings
warnings.filterwarnings('ignore')
from azure.storage.blob import BlobServiceClient
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine

## Connection String Load

In [78]:
def load_config_azure(config_path='config.json'):
    """Load Azure configuration parameters from config.json."""
    with open(config_path, 'r', encoding='utf-8') as config_file:
        config = json.load(config_file)
    return config['AZURE_CONNECTION_STRING'], config['CONTAINER_NAME']

def load_config_snowflake(config_path='config.json'):
    """Load Snowflake configuration parameters from config.json."""
    with open(config_path, 'r', encoding='utf-8') as config_file:
        config = json.load(config_file)
    return (
        config['SNOWFLAKE_USER'],
        config['SNOWFLAKE_PASSWORD'],
        config['SNOWFLAKE_ACCOUNT'],
        config['SNOWFLAKE_WAREHOUSE'],
        config['SNOWFLAKE_DATABASE'],
        config['SNOWFLAKE_SCHEMA']
    )

# Load Azure credentials
AZURE_CONNECTION_STRING, CONTAINER_NAME = load_config_azure('config.json')
blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)
print('Azure connected successfully')

# Load Snowflake credentials
SF_USER, SF_PASSWORD, SF_ACCOUNT, SF_WAREHOUSE, SF_DATABASE, SF_SCHEMA = load_config_snowflake('config.json')

engine = create_engine(URL(
    user=SF_USER,
    password=SF_PASSWORD,
    account=SF_ACCOUNT,
    warehouse=SF_WAREHOUSE,
    database=SF_DATABASE,
    schema=SF_SCHEMA
))
print('Snowflake connected successfully')

Azure connected successfully
Snowflake connected successfully


## Extraction

### Extraction - ZIP Code Population Data

In [79]:
# Extract zip.csv from Azure Blob Storage
blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob='zip.csv')
blob_data = blob_client.download_blob().readall()
zip_raw = pd.read_csv(io.BytesIO(blob_data), encoding='utf-8-sig')
print('ZIP raw shape:', zip_raw.shape)
zip_raw.head()

ZIP raw shape: (67548, 2)


,Label (Grouping),Total
0,ZCTA5 00601,NaN
1,ZCTA5 00601,"17,242"
2,ZCTA5 00602,NaN
3,ZCTA5 00602,"37,548"
4,ZCTA5 00603,NaN


### Extraction - State Population Data

In [80]:
# Extract state.csv from Azure Blob Storage
blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob='state.csv')
blob_data = blob_client.download_blob().readall()
state_raw = pd.read_csv(io.BytesIO(blob_data), encoding='utf-8-sig')
print('State raw shape:', state_raw.shape)
state_raw.head()

State raw shape: (104, 2)


,Label (Grouping),Total
0,Alabama,NaN
1,Alabama,"5,024,279"
2,Alaska,NaN
3,Alaska,"733,391"
4,Arizona,NaN


### Extraction - County Population Data

In [81]:
# Extract county.csv from Azure Blob Storage
blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob='county.csv')
blob_data = blob_client.download_blob().readall()
county_raw = pd.read_csv(io.BytesIO(blob_data), encoding='utf-8-sig')
print('County raw shape:', county_raw.shape)
county_raw.head()

County raw shape: (6444, 2)


,Label (Grouping),Total
0,"Autauga County, Alabama",NaN
1,Estimate,"59,285"
2,"Baldwin County, Alabama",NaN
3,Estimate,"239,945"
4,"Barbour County, Alabama",NaN


### Extraction - ZIP to County/State Crosswalk

In [82]:
# Download ZIP-to-county-to-state crosswalk from public source
import requests
crosswalk_url = 'https://raw.githubusercontent.com/scpike/us-state-county-zip/master/geo-data.csv'
r = requests.get(crosswalk_url)
crosswalk_raw = pd.read_csv(io.StringIO(r.text), dtype=str)
print('Crosswalk shape:', crosswalk_raw.shape)
crosswalk_raw.head()

Crosswalk shape: (33103, 6)


,state_fips,state,state_abbr,zipcode,county,city
0,1,Alabama,AL,35004,St. Clair,Acmar
1,1,Alabama,AL,35005,Jefferson,Adamsville
2,1,Alabama,AL,35006,Jefferson,Adger
3,1,Alabama,AL,35007,Shelby,Keystone
4,1,Alabama,AL,35010,Tallapoosa,New site


## Transformation

### Data Profiling

In [83]:
print('=== ZIP RAW ===')
print(zip_raw.shape)
print(zip_raw.info())
print(zip_raw.isnull().sum())

=== ZIP RAW ===
(67548, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67548 entries, 0 to 67547
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Label (Grouping)  67548 non-null  object
 1   Total             33774 non-null  object
dtypes: object(2)
memory usage: 1.0+ MB
None
Label (Grouping)        0
Total               33774
dtype: int64


In [84]:
print('=== STATE RAW ===')
print(state_raw.shape)
print(state_raw.info())
print(state_raw.isnull().sum())

=== STATE RAW ===
(104, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Label (Grouping)  104 non-null    object
 1   Total             52 non-null     object
dtypes: object(2)
memory usage: 1.8+ KB
None
Label (Grouping)     0
Total               52
dtype: int64


In [85]:
print('=== COUNTY RAW ===')
print(county_raw.shape)
print(county_raw.info())
print(county_raw.isnull().sum())

=== COUNTY RAW ===
(6444, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6444 entries, 0 to 6443
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Label (Grouping)  6444 non-null   object
 1   Total             3222 non-null   object
dtypes: object(2)
memory usage: 100.8+ KB
None
Label (Grouping)       0
Total               3222
dtype: int64


### Data Cleaning

In [86]:
# Clean ZIP population data
zip_raw.columns = ['label', 'population']

# Keep only rows that contain actual data (rows with ZCTA5 and a population value)
zip_clean = zip_raw[zip_raw['label'].str.contains('ZCTA5', na=False)].copy()
zip_clean = zip_clean[zip_clean['population'].notna() & (zip_clean['population'] != '')]

# Drop rows where population is missing
zip_clean = zip_clean.dropna(subset=['population'])
print('ZIP after cleaning:', zip_clean.shape)
zip_clean.head()

ZIP after cleaning: (33774, 2)


,label,population
1,ZCTA5 00601,"17,242"
3,ZCTA5 00602,"37,548"
5,ZCTA5 00603,"49,804"
7,ZCTA5 00606,"5,009"
9,ZCTA5 00610,"25,731"


In [87]:
# Clean State population data
state_raw.columns = ['label', 'population']

# Rows with \xa0 padding are data rows, rows without are state name headers
state_clean = state_raw[state_raw['label'].str.startswith('\xa0')].copy()
state_clean = state_clean[state_clean['population'].notna() & (state_clean['population'] != '')]
state_clean = state_clean.dropna(subset=['population'])
print('State after cleaning:', state_clean.shape)
state_clean.head()

State after cleaning: (52, 2)


,label,population
1,Alabama,"5,024,279"
3,Alaska,"733,391"
5,Arizona,"7,151,502"
7,Arkansas,"3,011,524"
9,California,"39,538,223"


In [88]:
# Clean County population data
county_raw.columns = ['label', 'population']

# Rows with \xa0 padding are data rows
county_clean = county_raw[county_raw['label'].str.startswith('\xa0')].copy()
county_clean = county_clean[county_clean['population'].notna() & (county_clean['population'] != '')]
county_clean = county_clean.dropna(subset=['population'])
print('County after cleaning:', county_clean.shape)
county_clean.head()

County after cleaning: (3222, 2)


,label,population
1,Estimate,"59,285"
3,Estimate,"239,945"
5,Estimate,"24,757"
7,Estimate,"22,152"
9,Estimate,"59,292"


### Data Reformatting

In [89]:
# Reformat ZIP data
zip_clean['zip_code'] = zip_clean['label'].str.extract(r'(\d{5})')
zip_clean['zip_code'] = zip_clean['zip_code'].astype(str).str.zfill(5)
zip_clean['zip_population'] = zip_clean['population'].str.replace(',', '').astype(int)
zip_clean = zip_clean[['zip_code', 'zip_population']]
print('ZIP reformatted:', zip_clean.shape)
zip_clean.head()

ZIP reformatted: (33774, 2)


,zip_code,zip_population
1,00601,17242
3,00602,37548
5,00603,49804
7,00606,5009
9,00610,25731


In [90]:
# Reformat State data
state_clean['state'] = state_clean['label'].str.replace('\xa0', '').str.strip()
state_clean['state_population'] = state_clean['population'].str.replace(',', '').astype(int)
state_clean = state_clean[['state', 'state_population']]
print('State reformatted:', state_clean.shape)
state_clean.head()

State reformatted: (52, 2)


,state,state_population
1,Alabama,5024279
3,Alaska,733391
5,Arizona,7151502
7,Arkansas,3011524
9,California,39538223


In [91]:
# Reformat County data
# County name is on row N, population is on row N+1 (labeled "Estimate")
county_raw_reset = county_raw.reset_index(drop=True)

counties = []
for i in range(0, len(county_raw_reset) - 1, 2):
    name_row = county_raw_reset.iloc[i]['label'].strip()
    pop_row = county_raw_reset.iloc[i + 1]['population']
    
    if ',' in name_row and str(pop_row).strip() not in ['', 'nan']:
        county = name_row.rsplit(',', 1)[0].strip()
        state = name_row.rsplit(',', 1)[1].strip()
        population = int(str(pop_row).replace(',', '').strip())
        counties.append({
            'county': county,
            'state': state,
            'county_population': population
        })

county_clean = pd.DataFrame(counties)
print('County reformatted:', county_clean.shape)
print(county_clean.head(10))

County reformatted: (3222, 3)
            county    state  county_population
0   Autauga County  Alabama              59285
1   Baldwin County  Alabama             239945
2   Barbour County  Alabama              24757
3      Bibb County  Alabama              22152
4    Blount County  Alabama              59292
5   Bullock County  Alabama              10157
6    Butler County  Alabama              18807
7   Calhoun County  Alabama             116141
8  Chambers County  Alabama              34450
9  Cherokee County  Alabama              25224


In [92]:
# Fix county name mismatch - remove " County" suffix to match crosswalk
county_clean['county'] = county_clean['county'].str.replace(' County', '', regex=False).str.strip()

# Also fix state name - county file has full state name, crosswalk also has full name so should match
# Check what both sides look like
print('County clean sample:')
print(county_clean.head())
print()
print('Crosswalk county sample:')
print(crosswalk_raw[['county', 'state']].head())
print()

# Check if any counties match after fix
matches = county_clean['county'].isin(crosswalk_raw['county']).sum()
print(f'Matching counties after fix: {matches}')

County clean sample:
    county    state  county_population
0  Autauga  Alabama              59285
1  Baldwin  Alabama             239945
2  Barbour  Alabama              24757
3     Bibb  Alabama              22152
4   Blount  Alabama              59292

Crosswalk county sample:
       county    state
0   St. Clair  Alabama
1   Jefferson  Alabama
2   Jefferson  Alabama
3      Shelby  Alabama
4  Tallapoosa  Alabama

Matching counties after fix: 3104


In [93]:
# Build dim_geography
merged = zip_clean.merge(crosswalk[['zip_code', 'state', 'state_abbr', 'county', 'city']], 
                         on='zip_code', how='left')

merged = merged.merge(state_clean, on='state', how='left')
merged = merged.merge(county_clean, on=['county', 'state'], how='left')
merged = merged.dropna(subset=['state'])

dim_geography = merged[[
    'zip_code', 'city', 'county', 'state', 'state_abbr',
    'zip_population', 'county_population', 'state_population'
]].drop_duplicates(subset=['zip_code']).reset_index(drop=True)

dim_geography.insert(0, 'geography_id', range(1, len(dim_geography) + 1))

print('dim_geography shape:', dim_geography.shape)
print('Null counts:')
print(dim_geography.isnull().sum())
print(dim_geography.head())

NameError: name 'crosswalk' is not defined

In [ ]:
# Check actual crosswalk column names
print(crosswalk_raw.columns.tolist())
print(crosswalk_raw.head(3))

['state_fips', 'state', 'state_abbr', 'zipcode', 'county', 'city']
  state_fips    state state_abbr zipcode     county        city
0          1  Alabama         AL   35004  St. Clair       Acmar
1          1  Alabama         AL   35005  Jefferson  Adamsville
2          1  Alabama         AL   35006  Jefferson       Adger


In [ ]:
# Reformat Crosswalk
crosswalk = crosswalk_raw.copy()
crosswalk['zipcode'] = crosswalk['zipcode'].astype(str).str.zfill(5)
crosswalk = crosswalk.rename(columns={'zipcode': 'zip_code'})
print('Crosswalk columns:', crosswalk.columns.tolist())
print('Crosswalk reformatted:', crosswalk.shape)
crosswalk.head()

KeyError: 'zipcode'

### Data Transformation - Build Dimension Tables

In [ ]:
# Build dim_geography
# Join zip population to crosswalk to get county/state per zip
merged = zip_clean.merge(crosswalk_raw[['zip_code', 'state', 'state_abbr', 'county', 'city']], 
                         on='zip_code', how='left')

# Join state population
merged = merged.merge(state_clean, on='state', how='left')

# Join county population
merged = merged.merge(county_clean, on=['county', 'state'], how='left')

# Drop non-US territories (Puerto Rico etc - no state match)
merged = merged.dropna(subset=['state'])

# Final dim_geography
dim_geography = merged[[
    'zip_code',
    'city',
    'county',
    'state',
    'state_abbr',
    'zip_population',
    'county_population',
    'state_population'
]].drop_duplicates(subset=['zip_code']).reset_index(drop=True)

# Add surrogate key
dim_geography.insert(0, 'geography_id', range(1, len(dim_geography) + 1))

print('dim_geography shape:', dim_geography.shape)
dim_geography.head()

dim_geography shape: (31409, 9)


,geography_id,zip_code,city,county,state,state_abbr,zip_population,county_population,state_population
0,1,01001,Agawam,Hampden,Massachusetts,MA,16984,NaN,7029917.0
1,2,01002,Cushman,Hampshire,Massachusetts,MA,27558,NaN,7029917.0
2,3,01005,Barre,Worcester,Massachusetts,MA,4900,NaN,7029917.0
3,4,01007,Belchertown,Hampshire,Massachusetts,MA,15423,NaN,7029917.0
4,5,01008,Blandford,Hampden,Massachusetts,MA,1317,NaN,7029917.0


### Data Consolidation

In [ ]:
# Final check on all dimension tables before loading
print('dim_geography:', dim_geography.shape)
print(dim_geography.isnull().sum())
dim_geography.head(10)

dim_geography: (31409, 9)
geography_id             0
zip_code                 0
city                    26
county                   0
state                    0
state_abbr               0
zip_population           0
county_population    30579
state_population      5461
dtype: int64


,geography_id,zip_code,city,county,state,state_abbr,zip_population,county_population,state_population
0,1,01001,Agawam,Hampden,Massachusetts,MA,16984,NaN,7029917.0
1,2,01002,Cushman,Hampshire,Massachusetts,MA,27558,NaN,7029917.0
2,3,01005,Barre,Worcester,Massachusetts,MA,4900,NaN,7029917.0
3,4,01007,Belchertown,Hampshire,Massachusetts,MA,15423,NaN,7029917.0
4,5,01008,Blandford,Hampden,Massachusetts,MA,1317,NaN,7029917.0
5,6,01010,Brimfield,Hampden,Massachusetts,MA,3711,NaN,7029917.0
6,7,01011,Chester,Hampden,Massachusetts,MA,1201,NaN,7029917.0
7,8,01012,Chesterfield,Hampshire,Massachusetts,MA,519,NaN,7029917.0
8,9,01013,Chicopee,Hampden,Massachusetts,MA,23656,NaN,7029917.0
9,10,01020,Chicopee,Hampden,Massachusetts,MA,29781,NaN,7029917.0


## Loading

In [ ]:
# Create table in Snowflake if it doesn't exist
create_table_sql = """
CREATE TABLE IF NOT EXISTS dim_geography (
    geography_id    INTEGER,
    zip_code        VARCHAR(10),
    city            VARCHAR(100),
    county          VARCHAR(100),
    state           VARCHAR(100),
    state_abbr      VARCHAR(5),
    zip_population  INTEGER,
    county_population INTEGER,
    state_population  INTEGER
);
"""
with engine.connect() as conn:
    conn.execute(create_table_sql)
print('Table created successfully')

DatabaseError: (snowflake.connector.errors.DatabaseError) 250001 (08001): Failed to connect to DB: BTZYFGU-TJ20285.snowflakecomputing.com:443. Incorrect username or password was specified.
(Background on this error at: https://sqlalche.me/e/20/4xp6)

In [ ]:
# Load dim_geography to Snowflake
chunksize = 5000
for i in range(0, dim_geography.shape[0], chunksize):
    chunk = dim_geography[i:i + chunksize]
    chunk.to_sql('dim_geography', engine, if_exists='append', index=False, method='multi')
    print(f'Loaded rows {i} to {i + len(chunk)}')

print('dim_geography loaded successfully!')

In [ ]:
# Verify load in Snowflake
result = pd.read_sql('SELECT COUNT(*) AS total FROM dim_geography', engine)
print('Total rows in Snowflake dim_geography:', result['total'][0])